# HGNN Neighbor-Sampled Training (Universal)

This notebook is configured for HGNN training across multiple environments (local, cloud, Streamlit) using configurable batch settings.

Outputs:
- `models/hgnn_model.pt`
- `outputs/hgnn/hgnn_training_curves.png`


In [ ]:
# Universal CUDA/GPU runtime configuration (works on local, cloud, Streamlit)
import os
import sys
from pathlib import Path

print('Notebook CWD:', Path.cwd().resolve())

def locate_project_root() -> Path:
    """
    Find folder containing src/config.py across multiple environments.
    Works on local machines, cloud platforms, and Streamlit.
    """
    # 1) Respect explicit env var first
    env_root = os.environ.get('PROJECT_ROOT')
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / 'src' / 'config.py').exists():
            return p

    # 2) Check current working directory and parents
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src' / 'config.py').exists():
            return candidate
    
    # 3) Check relative to notebook location
    notebook_dir = Path(__file__).resolve().parent if '__file__' in dir() else cwd
    for relative_path in ['..', '../..', '../../..']:
        candidate = (notebook_dir / relative_path).resolve()
        if (candidate / 'src' / 'config.py').exists():
            return candidate

    # 4) Common deployment paths
    deployment_candidates = [
        Path('/workspace/PROJECT'),
        Path('/workspace/project/PROJECT'),
        Path.home() / 'PROJECT',
    ]
    for p in deployment_candidates:
        if (p / 'src' / 'config.py').exists():
            return p

    raise RuntimeError(
        'Could not locate project root containing src/config.py. '
        'Set PROJECT_ROOT environment variable.'
    )

PROJECT_ROOT = locate_project_root()
os.environ['PROJECT_ROOT'] = str(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Universal HGNN settings (auto-adapts to available hardware)
os.environ['BUILD_HETERO_GRAPH'] = '1'
os.environ['USE_NEIGHBOR_SAMPLED_HGNN'] = '1'
os.environ['ALLOW_HGNN_FALLBACK'] = '1'  # Allow fallback if GPU unavailable

# CPU thread settings (conservative for cloud/Streamlit)
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '2')
os.environ.setdefault('SAMPLED_HGNN_TORCH_THREADS', '2')
os.environ.setdefault('SAMPLED_HGNN_TORCH_INTEROP_THREADS', '1')

# Memory-efficient loader settings
os.environ.setdefault('SAMPLED_HGNN_BATCH_SIZE', '256')
os.environ.setdefault('SAMPLED_HGNN_NEIGHBORS', '8,4')
os.environ.setdefault('SAMPLED_HGNN_NUM_WORKERS', '0')
os.environ.setdefault('SAMPLED_HGNN_PERSISTENT_WORKERS', '0')
os.environ.setdefault('SAMPLED_HGNN_PREFETCH_FACTOR', '2')
os.environ.setdefault('HGNN_MIN_FREE_GB', '5')

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print('BUILD_HETERO_GRAPH:', os.environ['BUILD_HETERO_GRAPH'])
print('USE_NEIGHBOR_SAMPLED_HGNN:', os.environ['USE_NEIGHBOR_SAMPLED_HGNN'])
print('ALLOW_HGNN_FALLBACK:', os.environ['ALLOW_HGNN_FALLBACK'])
print('SAMPLED_HGNN_NUM_WORKERS:', os.environ.get('SAMPLED_HGNN_NUM_WORKERS'))
print('SAMPLED_HGNN_TORCH_THREADS:', os.environ.get('SAMPLED_HGNN_TORCH_THREADS'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '<auto>'))


Notebook CWD: /jupyter/sods.capstone22
PROJECT_ROOT: /jupyter/sods.capstone22/PROJECT3
STRICT_GPU_ONLY: 1
USE_NEIGHBOR_SAMPLED_HGNN: 1
CUDA_VISIBLE_DEVICES: <all>
SAMPLED_HGNN_NUM_WORKERS: 1
SAMPLED_HGNN_TORCH_THREADS: 4


In [ ]:
import importlib.util
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import src.config as config

# Backward compatibility for config variants
OUTPUT_DIR = config.OUTPUT_DIR
MODEL_DIR = config.MODEL_DIR
HGNN_MODEL_PATH = getattr(config, 'HGNN_MODEL_PATH', os.path.join(str(MODEL_DIR), 'hgnn_model.pt'))

from src.data_loader import load_raw_data
from src.preprocessing import run_preprocessing_pipeline
from src.training import train_neighbor_sampled_hgnn

# Keep CPU thread usage bounded inside this kernel
torch.set_num_threads(int(os.environ.get('SAMPLED_HGNN_TORCH_THREADS', os.environ.get('OMP_NUM_THREADS', '2'))))
try:
    torch.set_num_interop_threads(int(os.environ.get('SAMPLED_HGNN_TORCH_INTEROP_THREADS', '1')))
except RuntimeError:
    pass

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    print('GPU 0:', torch.cuda.get_device_name(0))
else:
    print('Using CPU for training')

print('pyg_lib available:', bool(importlib.util.find_spec('pyg_lib')))
print('torch_sparse available:', bool(importlib.util.find_spec('torch_sparse')))
print('HGNN_MODEL_PATH:', HGNN_MODEL_PATH)
print('torch num threads:', torch.get_num_threads())


Torch: 2.6.0+cu124
CUDA available: True
CUDA device count: 8
GPU 0: NVIDIA A100-SXM4-80GB
pyg_lib available: True
torch_sparse available: True
HGNN_MODEL_PATH: /jupyter/sods.capstone22/PROJECT3/models/hgnn_model.pt
torch num threads: 4


In [3]:
# Load full data and run preprocessing pipeline (builds hetero graph)
df = load_raw_data(sample_frac=None)
print('Raw shape:', df.shape)

prep = run_preprocessing_pipeline(df)
hetero_data = prep.get('hetero_data')
if hetero_data is None:
    raise RuntimeError('hetero_data is missing. Ensure BUILD_HETERO_GRAPH=1 before preprocessing.')

print('Train shape:', prep['X_train'].shape)
print('Val shape:', prep['X_val'].shape)
print('Test shape:', prep['X_test'].shape)
print('Train fraud ratio:', float(prep['y_train'].mean()))

2026-04-10 22:45:17 | DataLoader           | INFO    | ⏳ Starting: Loading transaction data
2026-04-10 22:45:28 | DataLoader           | INFO    |   Transactions: 590,540 rows × 394 cols
2026-04-10 22:45:28 | DataLoader           | INFO    | ✅ Finished: Loading transaction data (10.3s)
2026-04-10 22:45:28 | DataLoader           | INFO    | ⏳ Starting: Loading identity data
2026-04-10 22:45:28 | DataLoader           | INFO    |   Identity:     144,233 rows × 41 cols
2026-04-10 22:45:28 | DataLoader           | INFO    | ✅ Finished: Loading identity data (0.3s)
2026-04-10 22:45:28 | DataLoader           | INFO    | ⏳ Starting: Merging datasets on TransactionID
2026-04-10 22:45:28 | DataLoader           | INFO    |   Merged:       590,540 rows × 434 cols
2026-04-10 22:45:28 | DataLoader           | INFO    | ✅ Finished: Merging datasets on TransactionID (0.4s)
2026-04-10 22:45:28 | DataLoader           | INFO    | ⏳ Starting: Optimizing memory usage
2026-04-10 22:45:37 | DataLoader       

  Memory: 2567.1 MB → 1656.4 MB (35.5% reduction)
Raw shape: (590540, 434)


2026-04-10 22:45:38 | Preprocessing        | INFO    |   Dropped 214 columns with >50% missing
2026-04-10 22:45:40 | Preprocessing        | INFO    |   Features: 434 → 220 (removed 214)
2026-04-10 22:45:40 | Preprocessing        | INFO    |   Remaining nulls: 0
2026-04-10 22:45:40 | Preprocessing        | INFO    | ✅ Finished: Handling missing values (3.1s)
2026-04-10 22:45:40 | Preprocessing        | INFO    | ⏳ Starting: Capping outliers
2026-04-10 22:45:40 | Preprocessing        | INFO    |   Capped 5,641 outliers in TransactionAmt at 1104.00
2026-04-10 22:45:40 | Preprocessing        | INFO    | ✅ Finished: Capping outliers (0.0s)
2026-04-10 22:45:40 | Preprocessing        | INFO    | ⏳ Starting: Encoding categoricals
2026-04-10 22:45:41 | Preprocessing        | INFO    |   Encoded 9 categorical columns
2026-04-10 22:45:41 | Preprocessing        | INFO    | ✅ Finished: Encoding categoricals (0.7s)
2026-04-10 22:45:41 | Preprocessing        | INFO    | ⏳ Starting: Splitting data (70

Train shape: (413378, 217)
Val shape: (88581, 217)
Test shape: (88581, 217)
Train fraud ratio: 0.034989767234831076


In [4]:
# Confirm full-dataset graph stats
num_tx = int(hetero_data['transaction'].x.shape[0])
num_edges = {str(k): int(v.edge_index.shape[1]) for k, v in hetero_data.edge_items()}
print('Transactions in graph:', num_tx)
print('Edge counts by relation:')
for rel, count in num_edges.items():
    print(' ', rel, '->', count)

print('Batch settings:')
print(' SAMPLED_HGNN_BATCH_SIZE =', os.environ.get('SAMPLED_HGNN_BATCH_SIZE'))
print(' SAMPLED_HGNN_NEIGHBORS  =', os.environ.get('SAMPLED_HGNN_NEIGHBORS'))

Transactions in graph: 590540
Edge counts by relation:
  ('transaction', 'used', 'card') -> 590540
  ('card', 'used_by', 'transaction') -> 590540
  ('transaction', 'on', 'device') -> 590540
  ('device', 'hosts', 'transaction') -> 590540
Batch settings:
 SAMPLED_HGNN_BATCH_SIZE = 512
 SAMPLED_HGNN_NEIGHBORS  = 10,6


In [5]:
# Train neighbor-sampled HGNN on full dataset (DGX CUDA batch training with adaptive backoff)
import traceback
import importlib
import subprocess
import src.training as training_mod

# Ensure latest training code is loaded in long-lived notebook kernels.
training_mod = importlib.reload(training_mod)
train_neighbor_sampled_hgnn = training_mod.train_neighbor_sampled_hgnn

if not torch.cuda.is_available():
    raise RuntimeError(
        'DGX profile requires CUDA. Set CUDA_VISIBLE_DEVICES to a free GPU and rerun.'
    )

def _pick_free_gpu() -> int:
    """Pick the least-busy GPU using nvidia-smi metrics."""
    cmd = [
        'nvidia-smi',
        '--query-gpu=index,memory.used,memory.total,utilization.gpu',
        '--format=csv,noheader,nounits',
    ]
    out = subprocess.check_output(cmd, text=True).strip().splitlines()
    candidates = []
    for line in out:
        idx_s, used_s, total_s, util_s = [x.strip() for x in line.split(',')]
        idx = int(idx_s)
        used = float(used_s)
        total = float(total_s)
        util = float(util_s)
        used_pct = used / max(total, 1.0)
        score = used_pct * 100.0 + util
        candidates.append((score, idx, used, total, util))
    candidates.sort(key=lambda x: x[0])
    best = candidates[0]
    print(f'Auto-selected GPU {best[1]} (used={best[2]:.0f}MB/{best[3]:.0f}MB, util={best[4]:.0f}%)')
    return int(best[1])

def _set_active_gpu(gpu_id: int) -> None:
    torch.cuda.set_device(gpu_id)
    os.environ['ACTIVE_CUDA_DEVICE'] = str(gpu_id)

manual_gpu = os.environ.get('DGX_GPU_ID')
if manual_gpu is not None and manual_gpu.strip() != '':
    chosen_gpu = int(manual_gpu)
    print('Using DGX_GPU_ID override:', chosen_gpu)
else:
    chosen_gpu = _pick_free_gpu()

_set_active_gpu(chosen_gpu)
print('Current CUDA device index:', torch.cuda.current_device())
print('Current CUDA device name:', torch.cuda.get_device_name(torch.cuda.current_device()))

batch_schedule = [
    int(os.environ.get('SAMPLED_HGNN_BATCH_SIZE', '512')),
    384,
    256,
    192,
    128,
 ]
neighbor_schedule = [
    os.environ.get('SAMPLED_HGNN_NEIGHBORS', '10,6'),
    '8,5',
    '6,4',
 ]

# Deduplicate while preserving order.
batch_schedule = list(dict.fromkeys([b for b in batch_schedule if b > 0]))
neighbor_schedule = list(dict.fromkeys(neighbor_schedule))

print('Adaptive schedules:')
print(' batch_schedule   =', batch_schedule)
print(' neighbor_schedule=', neighbor_schedule)
print(' CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<all>'))
print(' ACTIVE_CUDA_DEVICE  =', os.environ.get('ACTIVE_CUDA_DEVICE', '<unset>'))
print(' SAMPLED_HGNN_NUM_WORKERS =', os.environ.get('SAMPLED_HGNN_NUM_WORKERS'))

last_error = None
nn_model, nn_history = None, None
trained = False

for neigh in neighbor_schedule:
    os.environ['SAMPLED_HGNN_NEIGHBORS'] = neigh
    for bs in batch_schedule:
        os.environ['SAMPLED_HGNN_BATCH_SIZE'] = str(bs)
        try:
            print(f'\nAttempt: neighbors={neigh}, batch_size={bs}, gpu={torch.cuda.current_device()}')
            nn_model, nn_history = train_neighbor_sampled_hgnn(
                X_train_df=prep['X_train'],
                y_train_df=prep['y_train'],
                X_val=prep['X_val'],
                y_val=prep['y_val'],
                hetero_data=hetero_data,
                use_smote=False,
                allow_cpu_retry=False,
            )
            trained = True
            break
        except RuntimeError as exc:
            last_error = exc
            msg = str(exc).lower()
            is_mem = ('out of memory' in msg) or ('cuda' in msg and 'memory' in msg)
            is_busy = ('cuda-capable device' in msg and ('busy' in msg or 'unavailable' in msg))

            if is_busy:
                print('GPU busy/unavailable detected; attempting one-time GPU re-selection...')
                try:
                    chosen_gpu = _pick_free_gpu()
                    _set_active_gpu(chosen_gpu)
                    print('Switched active GPU to:', torch.cuda.current_device())
                    continue
                except Exception as switch_exc:
                    raise RuntimeError(
                        f'GPU is busy/unavailable and auto-switch failed: {switch_exc}. '
                        'Reserve a free DGX GPU and set DGX_GPU_ID / CUDA_VISIBLE_DEVICES.'
                    ) from exc

            if is_mem:
                print(f'Backoff triggered for neighbors={neigh}, batch_size={bs}: {exc}')
                continue
            raise
    if trained:
        break

if not trained:
    traceback.print_exception(type(last_error), last_error, last_error.__traceback__)
    raise RuntimeError(
        'Neighbor-sampled HGNN failed after adaptive backoff. '
        'Reduce SAMPLED_HGNN_BATCH_SIZE and SAMPLED_HGNN_NEIGHBORS further (for example 96 and 5,3).'
)

print('Training completed.')
print('Best val AUC from history:', max(nn_history.get('val_aucs', [0.0])))
print('Model saved to:', HGNN_MODEL_PATH)

2026-04-10 22:45:50 | Training             | INFO    | ==================================================
2026-04-10 22:45:50 | Training             | INFO    |   🧠 Training Neural Network (Neighbor-Sampled HGNN)
2026-04-10 22:45:50 | Training             | INFO    | ==================================================
2026-04-10 22:45:50 | Training             | INFO    |   Device: cuda
2026-04-10 22:45:50 | Training             | INFO    |   CUDA free memory: 73.67 / 79.15 GB
2026-04-10 22:45:50 | Models               | INFO    |   🧠 FraudHGNN created (1,898,279 parameters)
2026-04-10 22:45:50 | Models               | INFO    |      Architecture: HGTConv Layers
2026-04-10 22:45:50 | Models               | INFO    |      Loss: FocalLoss(gamma=2.0, alpha=0.75)
2026-04-10 22:45:51 | Training             | INFO    | ⏳ Starting: Neural Network training (Neighbor-Sampled HGNN)
INFO:Training:⏳ Starting: Neural Network training (Neighbor-Sampled HGNN)


Auto-selected GPU 6 (used=5192MB/81920MB, util=0%)
Current CUDA device index: 6
Current CUDA device name: NVIDIA A100-SXM4-80GB
Adaptive schedules:
 batch_schedule   = [512, 384, 256, 192, 128]
 neighbor_schedule= ['10,6', '8,5', '6,4']
 CUDA_VISIBLE_DEVICES = <all>
 ACTIVE_CUDA_DEVICE  = 6
 SAMPLED_HGNN_NUM_WORKERS = 1

Attempt: neighbors=10,6, batch_size=512, gpu=6


2026-04-10 22:46:06 | Training             | INFO    |   Epoch   1/50 | Loss: 0.0179 | Val AUC: 0.8450
INFO:Training:  Epoch   1/50 | Loss: 0.0179 | Val AUC: 0.8450
2026-04-10 22:47:03 | Training             | INFO    |   Epoch   5/50 | Loss: 0.0151 | Val AUC: 0.8728
INFO:Training:  Epoch   5/50 | Loss: 0.0151 | Val AUC: 0.8728
2026-04-10 22:48:14 | Training             | INFO    |   Epoch  10/50 | Loss: 0.0150 | Val AUC: 0.8732
INFO:Training:  Epoch  10/50 | Loss: 0.0150 | Val AUC: 0.8732
2026-04-10 22:49:28 | Training             | INFO    |   Epoch  15/50 | Loss: 0.0145 | Val AUC: 0.8802
INFO:Training:  Epoch  15/50 | Loss: 0.0145 | Val AUC: 0.8802
2026-04-10 22:50:42 | Training             | INFO    |   Epoch  20/50 | Loss: 0.0250 | Val AUC: 0.8921
INFO:Training:  Epoch  20/50 | Loss: 0.0250 | Val AUC: 0.8921
2026-04-10 22:51:53 | Training             | INFO    |   Epoch  25/50 | Loss: 0.0124 | Val AUC: 0.8958
INFO:Training:  Epoch  25/50 | Loss: 0.0124 | Val AUC: 0.8958
2026-04-10

Training completed.
Best val AUC from history: 0.9121286431883938
Model saved to: /jupyter/sods.capstone22/PROJECT3/models/hgnn_model.pt


In [6]:
# Save training curves
hgnn_out = Path(OUTPUT_DIR) / 'hgnn'
hgnn_out.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(nn_history.get('train_losses', []), linewidth=2)
axes[0].set_title('Neighbor-Sampled HGNN Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(nn_history.get('val_aucs', []), linewidth=2)
axes[1].set_title('Neighbor-Sampled HGNN Validation ROC-AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC')
axes[1].grid(alpha=0.3)

fig.tight_layout()
curve_path = hgnn_out / 'dgx_hgnn_training_curves.png'
fig.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()

print('Saved curves to:', curve_path)

Saved curves to: /jupyter/sods.capstone22/PROJECT3/outputs/hgnn/dgx_hgnn_training_curves.png


## Notes for DGX Full-Data Training

- This notebook uses neighbor-sampled HGNN mini-batches for full-node coverage on DGX.
- Reserve a free CUDA device first, then set CUDA_VISIBLE_DEVICES before running.
- Use adaptive backoff: batch size and neighbors are automatically reduced on memory pressure.
- Saved model path is `models/hgnn_model.pt`.